# Virelion-DCCP — Safe Colab Runtime Validation

This notebook tests the checked-out DCCP source for actual runtime and integration defects.

**Safety:** it does not modify Colab's global Python environment. Do not upgrade or replace global `pip`, `setuptools`, `wheel`, Torch, or other runtime packages. All test dependencies are installed in an isolated dependency directory under `/content/dccp-test-env`.

**GPU/T4:** not required for the current DCCP codebase.

In [ ]:
# 1. Clone current main without changing the runtime environment
from pathlib import Path
import shutil, subprocess, sys, os

ROOT = Path("/content/Virelion-DCCP")
if ROOT.exists():
    shutil.rmtree(ROOT)

subprocess.run([
    "git", "clone", "--branch", "main", "--depth", "1",
    "https://github.com/Virelion-Biotech/Virelion-DCCP.git", str(ROOT)
], check=True)

COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True
).strip()

print("Tested commit:", COMMIT)
print("Colab/system Python:", sys.executable)


In [ ]:
# 2. Create an isolated dependency directory.
# IMPORTANT: this does NOT create a venv and does NOT modify Colab's installed packages.
DEPS = Path("/content/dccp-test-deps")
if DEPS.exists():
    shutil.rmtree(DEPS)
DEPS.mkdir(parents=True)

# Install validation dependencies ONLY into /content/dccp-test-deps.
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--disable-pip-version-check",
    "--no-warn-script-location",
    "--target", str(DEPS),
    "jsonschema", "pytest", "pytest-cov", "coverage",
    "hypothesis", "ruff", "pip-audit"
], check=True)

print("Isolated dependency directory:", DEPS)
print("Global Colab packages were not upgraded or replaced.")

In [ ]:
# 3. Run DCCP with the checkout + isolated dependency directory prepended.
ENV = os.environ.copy()
ENV["PYTHONPATH"] = os.pathsep.join([str(DEPS), str(ROOT / "src")])
ENV["PYTHONNOUSERSITE"] = "1"

def run(*args, check=True, capture=False):
    cmd = [str(Path(sys.executable)), *map(str, args)]
    return subprocess.run(
        cmd, cwd=ROOT, env=ENV, check=check,
        text=True, capture_output=capture
    )

def show(result):
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:\n", result.stderr)

p = run("-c", "import dccp; print(dccp.__file__)", capture=True)
show(p)
assert p.returncode == 0
assert str(ROOT / "src") in p.stdout
print("✓ DCCP source import is coming from this checkout")

In [ ]:
# 4. Compile, Ruff, and pytest + coverage
run("-m", "compileall", "-q", "src")
run("-m", "ruff", "check", ".")
run(
    "-m", "pytest", "-q",
    "--cov=dccp", "--cov-branch",
    "--cov-report=term-missing", "--cov-report=html"
)
print("Baseline static/runtime suite: PASS");


In [ ]:
# 5. Import every DCCP module in a fresh subprocess
import pathlib, subprocess, sys, traceback

modules = sorted((ROOT / "src" / "dccp").glob("*.py"))
failures = []

for path in modules:
    if path.name == "__init__.py":
        continue
    module = path.stem
    p = subprocess.run(
        [sys.executable, "-c", f"import dccp.{module}"],
        cwd=ROOT, text=True, capture_output=True
    )
    print(f"dccp.{module}:", "OK" if p.returncode == 0 else "FAIL")
    if p.returncode:
        failures.append((module, p.stdout, p.stderr))

if failures:
    for module, stdout, stderr in failures:
        print(f"\n--- {module} ---\n{stdout}\n{stderr}")
raise AssertionError(f"{len(failures)} module import failures")


In [ ]:
# 6. CLI smoke tests through the Python entry point
import subprocess, sys

commands = [
    [sys.executable, "-m", "dccp.cli", "--help"],
    [sys.executable, "-m", "dccp.cli", "audit-all", "scenarios"],
]

for cmd in commands:
    p = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
    print("$", " ".join(cmd))
    print("return code:", p.returncode)
    if p.stdout:
        print(p.stdout[:12000])
    if p.stderr:
        print("STDERR:", p.stderr[:12000])
    assert p.returncode == 0


In [ ]:
# 7. Load and audit EVERY scenario individually
from dccp.scenario import load_scenario
from dccp.audit import audit_scenario

scenario_root = ROOT / "scenarios"
scenario_files = sorted(scenario_root.rglob("*.json"))
print("Scenario files:", len(scenario_files))
assert scenario_files

failures = []
for path in scenario_files:
    rel = path.relative_to(ROOT)
    try:
        scenario = load_scenario(path)
        result = audit_scenario(scenario)
        print(rel, "->", "PASS" if result.passed else "FAIL")
        if not result.passed:
            failures.append({
                "path": str(rel),
                "schema_errors": result.schema_errors,
                "policy_errors": result.policy_errors,
                "warnings": result.policy_warnings,
            })
    except Exception as exc:
        print(rel, "-> EXCEPTION:", repr(exc))
        failures.append({"path": str(rel), "exception": repr(exc)})

if failures:
    import pprint
    pprint.pp(failures)
raise AssertionError(f"{len(failures)} scenario failures")


In [ ]:
# 8. Registry / library / challenge-set / bundle integration
import json
import tempfile
from dccp.registry import build_registry, write_registry
from dccp.library import (discover_scenarios, load_library, materialize_challenge_set, write_challenge_set)
from dccp.bundle import build_bundle

with tempfile.TemporaryDirectory() as td:
    tmp = Path(td)
    registry_path = tmp / "registry.json"

    registry = build_registry(scenario_root, exclude_paths=[registry_path])
    print("Registry entries:", len(registry))
    assert registry

    written = write_registry(scenario_root, registry_path)
    assert registry_path.exists()
    assert written["n_entries"] == len(registry)
    loaded_registry = json.loads(registry_path.read_text(encoding="utf-8"))
    assert loaded_registry["n_entries"] == len(registry)
    assert all(e["path"] != registry_path.name for e in loaded_registry["entries"])
    print("Registry write/reload: PASS")

    discovered = discover_scenarios(scenario_root)
    library = load_library(scenario_root)
    assert len(discovered) == len(registry) == len(library)
    print("Discovery/library: PASS")

    challenge = materialize_challenge_set(scenario_root)
    assert challenge["n_cases"] == len(library)
    assert len(challenge["set_hash"]) == 64
    print("Challenge-set materialization: PASS")

    challenge_path = tmp / "challenge_set.json"
    write_challenge_set(challenge_path, scenario_root)
    reloaded_challenge = json.loads(challenge_path.read_text(encoding="utf-8"))
    assert reloaded_challenge["set_hash"] == challenge["set_hash"]
    print("Challenge-set write/reload: PASS")

    bundle_dir = tmp / "bundle"
    bundle = build_bundle(
        bundle_dir,
        run_id="colab-runtime-test",
        input_files=[discovered[0], discovered[0]],
        producer_version="0.3.0",
        base_dir=ROOT,
    )
    assert len(bundle["inputs"]) == 1
    assert not bundle["inputs"][0]["path"].startswith("/")
    assert (bundle_dir / "manifest.json").exists()
    print("Bundle generation/deduplication: PASS")

print("Infrastructure integration: PASS")


In [ ]:
# 9. Path traversal / unusual path handling
from dccp.registry import build_registry
from dccp.bundle import build_bundle

with tempfile.TemporaryDirectory() as td:
    root = Path(td)
    scenarios = root / "scenarios"
    scenarios.mkdir()

    for name in ["normal.json", "space name.json", "unicode-α.json", "nested/deep/scenario.json"]:
        p = scenarios / name
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text("{}", encoding="utf-8")

    # Registry should at least traverse the filesystem without path errors.
    entries = build_registry(scenarios)
    assert len(entries) == 4
    print("Unusual registry paths: PASS")

    outside = root.parent / "dccp_outside_colab_test.txt"
    outside.write_text("outside", encoding="utf-8")
    try:
        try:
            build_bundle(
                root / "bundle",
                run_id="outside-test",
                input_files=[outside],
                base_dir=scenarios,
            )
        except (ValueError, FileNotFoundError, RuntimeError):
            print("Outside-base rejection: PASS")
        else:
            raise AssertionError("Bundle accepted input outside base_dir")
    finally:
        outside.unlink(missing_ok=True)


In [ ]:
# 10. Non-finite numeric serialization
from dccp.fingerprint import canonical_json

for value in [float("nan"), float("inf"), float("-inf")]:
    try:
        canonical_json({"x": value})
    except (ValueError, TypeError):
        print(value, "-> rejected")
    else:
        raise AssertionError(f"canonical_json accepted {value}")

print("Non-finite protection: PASS")


In [ ]:
# 11. Hypothesis property test for deterministic fingerprinting
from hypothesis import given, strategies as st
from dccp.fingerprint import canonical_json, fingerprint

finite = st.floats(allow_nan=False, allow_infinity=False, width=64)

@given(st.dictionaries(st.text(max_size=30), finite, max_size=20))
def test_determinism(obj):
    assert canonical_json(obj) == canonical_json(obj)
    assert fingerprint(obj) == fingerprint(obj)

test_determinism()
print("Hypothesis determinism test: PASS")


In [ ]:
# 12. Clean wheel build + install test without a venv or global installation.
WHEEL_DIR = Path("/content/dccp-wheeltest")
WHEEL_INSTALL = Path("/content/dccp-wheel-install")
for path in (WHEEL_DIR, WHEEL_INSTALL):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True)

subprocess.run([
    str(sys.executable), "-m", "pip", "wheel",
    "--no-deps", str(ROOT), "-w", str(WHEEL_DIR)
], check=True, cwd=ROOT)

wheel = next(WHEEL_DIR.glob("*.whl"))

# Install the wheel into another target directory only.
subprocess.run([
    str(sys.executable), "-m", "pip", "install",
    "--disable-pip-version-check",
    "--no-warn-script-location",
    "--target", str(WHEEL_INSTALL),
    "--no-deps", str(wheel)
], check=True)

wheel_env = os.environ.copy()
wheel_env["PYTHONPATH"] = os.pathsep.join([
    str(WHEEL_INSTALL),
    str(DEPS),
])
wheel_env["PYTHONNOUSERSITE"] = "1"

p = subprocess.run(
    [str(sys.executable), "-c", "import dccp; print(dccp.__file__)"],
    cwd=ROOT, env=wheel_env, text=True,
    capture_output=True, check=True
)
print(p.stdout)
assert str(WHEEL_INSTALL) in p.stdout
print("✓ Clean wheel installation/import passed without modifying Colab")

In [ ]:
# 13. Public API smoke test
import dccp
public = [name for name in dir(dccp) if not name.startswith("_")]
assert public
for name in public:
    assert getattr(dccp, name) is not None
print("Resolved public exports:", len(public))
print("Public API smoke test: PASS")


In [ ]:
# 14. Dependency/security audit in the isolated environment
p = run("-m", "pip_audit", check=False, capture=True)
show(p)

if p.returncode == 0:
    print("pip-audit: PASS")
else:
    print("pip-audit reported findings or could not complete; review the output above.")


In [ ]:
# 15. Repeat the actual pytest suite to expose flaky/stateful failures
N = 10
for i in range(N):
    p = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    status = "PASS" if p.returncode == 0 else "FAIL"
    print(f"pytest run {i + 1}/{N}: {status}")
    if p.returncode:
        print(p.stdout)
        print(p.stderr)
        raise AssertionError(f"Repeated pytest run {i + 1} failed")

print(f"{N} repeated pytest runs: PASS")


In [ ]:
# 16. Repeated library-load determinism
from dccp.provenance import canonical_hash

observed = []
for _ in range(50):
    lib = load_library(scenario_root)
    observed.append(canonical_hash({
        "count": len(lib),
        "ids": sorted(entry.scenario.scenario_id for entry in lib),
        "digests": sorted(entry.digest for entry in lib),
    }))

assert len(set(observed)) == 1
print("50 repeated library loads: deterministic")


In [ ]:
# 17. Suspicious-pattern scan (review only; findings are not automatically failures)
import re

patterns = {
    "bare_except": r"except\s*:",
    "eval": r"\beval\s*\(",
    "exec": r"\bexec\s*\(",
    "shell_true": r"shell\s*=\s*True",
    "mutable_default": r"def\s+\w+\([^)]*=\s*(?:\[\]|\{\})",
    "todo_or_fixme": r"\b(?:TODO|FIXME)\b",
}

for name, pattern in patterns.items():
    hits = []
    for path in (ROOT / "src" / "dccp").rglob("*.py"):
        for lineno, line in enumerate(path.read_text(errors="replace").splitlines(), 1):
            if re.search(pattern, line):
                hits.append(f"{path.relative_to(ROOT)}:{lineno}: {line.strip()}")
    print(f"\n{name}: {len(hits)}")
    for hit in hits[:50]:
        print(" ", hit)

print("Pattern scan complete — manually inspect reported lines.")


In [ ]:
# 18. Dependency/security audit in the isolated environment
p = run("-m", "pip_audit", check=False, capture=True)
show(p)
if p.returncode == 0:
    print("pip-audit: PASS")
else:
    print("pip-audit reported findings or could not complete; review the output above.")


In [ ]:
# 19. Save a machine-readable runtime report
from datetime import datetime, timezone
import json

report = {
    "tested_at_utc": datetime.now(timezone.utc).isoformat(),
    "commit": COMMIT,
    "colab_python": sys.version,
    "test_python": subprocess.check_output([str(Path(sys.executable)), "--version"], text=True).strip(),
    "dependency_dir": str(DEPS),
    "note": "All DCCP tests used the isolated dependency directory; Colab global pip/setuptools/Torch were not upgraded."
}

report_path = ROOT / "colab_runtime_report.json"
report_path.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2))
print("Report:", report_path);

## Important

This notebook deliberately avoids editable installation into the Colab kernel and never upgrades the global `pip`/`setuptools`/Torch stack.

All validation dependencies live in an isolated dependency directory. The DCCP source is tested directly from the cloned checkout via `PYTHONPATH=src`.

The notebook is CPU-oriented; a T4 is not required for the current DCCP architecture.